# GNSS Orbit Determination: Extended Kalman Filter Execution

This notebook orchestrates the high-fidelity Extended Kalman Filter (EKF) simulation. It loads the truth trajectories, initializes the physical environment, pre-aligns the GNSS constellations, and executes the asynchronous measurement and time-update loops.

In [ ]:
import numpy as np
import csv
from pathlib import Path
from datetime import datetime
from scipy.interpolate import interp1d

# Import our modular Orbit Determination package
from orbit_determination.ekf_environment import Environment
from orbit_determination.ekf_spacecraft import Spacecraft
from orbit_determination.ekf_gnss import NavSatSystems
from orbit_determination.ekf import OrbitalEKF
from orbit_determination.ekf_simulation import SimulationManager
from orbit_determination.time_utils import julian_to_datetime

C_LIGHT = 299792458.0

In [ ]:
def setup_mass_and_thrust(t_data_array, time_from_eor_start_s, init_mass, fthr, Isp, ignition_data, fthr_dir):
    """Creates interpolators for spacecraft mass, engine ignition, and thrust vectors."""
    G = 9.8
    mass_values = [init_mass]
    mass = init_mass
    for i, data in enumerate(ignition_data[:-1]):
        if data[1] == 1:
            mass -= fthr/(G*Isp) * (ignition_data[i+1, 0] - data[0]) 
        mass_values.append(mass)

    t_ignition = ignition_data[:,0] - time_from_eor_start_s
    mass_func = interp1d(t_ignition, mass_values, bounds_error=False)
    ignition_func = interp1d(t_ignition, ignition_data[:,1], kind='previous', bounds_error=False)
    fthr_dir_func = interp1d(t_data_array, fthr_dir, axis=0, bounds_error=False)

    return mass_func, ignition_func, fthr_dir_func

In [ ]:
# Define paths based on the new project structure
project_root = Path.cwd().parent
data_path = project_root / 'data'

rcver_path = data_path / 'raw' / 'receiver_orbits' / 'eor' / 'sample2.txt'
ignition_path = data_path / 'raw' / 'receiver_orbits' / 'eor' / 'obt_ignition.txt'
clock_path = data_path / 'interim' / 'clock_sim' / 'true_clock_bias_drift.npy'

gnss_pos_path = data_path / 'interim' / 'ephemeris'
gnss_vis_path = data_path / 'processed' / 'visibility' / 'sample2'
export_path = data_path / 'processed' / 'filter_results' / 'sample2'

# Reference receiver trajectory starting date 
eor_start_date = datetime(2027, 3, 21, 5, 0, 0)

In [ ]:
# 1. Load Receiver Trajectory
rcver_data = np.loadtxt(rcver_path, skiprows=1, usecols=(0,13,14,15,16,17,18,28,29,30)).T

dates_utc = np.array([julian_to_datetime(jd) for jd in rcver_data[0]])
t_data_array = np.array([(date - dates_utc[0]).total_seconds() for date in dates_utc])

rcver_pos = 1e3 * np.column_stack((rcver_data[1], rcver_data[2], rcver_data[3]))
rcver_vel = 1e3 * np.column_stack((rcver_data[4], rcver_data[5], rcver_data[6]))
fthr_dir = np.column_stack((rcver_data[7], rcver_data[8], rcver_data[9]))

# 2. Load Clock States
clock_bias_drift = np.load(clock_path)
clock_bias = clock_bias_drift[0,:]
clock_drift = clock_bias_drift[1,:]

# 3. Setup Mass & Thrust
ignition_data = np.loadtxt(ignition_path, skiprows=7)
ignition_data[:,0] = ignition_data[:,0] - ignition_data[0,0]

mass_props = {'mass': 2883.5, 'Isp': 1670.32, 'thrust': 0.3745, 'area': 24.6, 'Cr': 1.3}

# Reference receiver trajectory starting date 
# Considering the reference satellite is undergoing an orbit raising,
# this is necessary to determine the appropiate mass and thrust profiles
# in the simulation time window considered.
eor_start_date = datetime(2027, 3, 21, 5, 0, 0)
eor_time_offset_s = (dates_utc[0] - eor_start_date).total_seconds()

mass_func, ignition_func, fthr_dir_func = setup_mass_and_thrust(
    t_data_array, eor_time_offset_s, mass_props['mass'], mass_props['thrust'], 
    mass_props['Isp'], ignition_data, fthr_dir
)

# 4. Initialize EKF Parameters (Noise & Apriori State)
first_fix_pos = 100 / np.sqrt(3)
first_fix_vel = 0.01 / np.sqrt(3)
first_fix_bias = C_LIGHT * 1e-7 
first_fix_drift = 0.01

x0 = np.concatenate((
    rcver_pos[0] + np.random.normal(scale=first_fix_pos, size=3),
    rcver_vel[0] + np.random.normal(scale=first_fix_vel, size=3),
    [clock_bias[0] + np.random.normal(scale=first_fix_bias)],
    [clock_drift[0] + np.random.normal(scale=first_fix_drift)]
))

P0 = np.diag(
    [first_fix_pos**2]*3 + 
    [first_fix_vel**2]*3 + 
    [first_fix_bias**2] + 
    [first_fix_drift**2]
)

# Acceleration white noise PSD
Sa = 3e-12

# Allan variance for OCXO clocks
h0, h1, h2 = 2e-25, 7e-25, 6e-25
noise_params = {
    'Sa': Sa,             # Acceleration white noise [m/s^2]
    'Sf': h0 / 2,            # Clock phase noise
    'Sg': 2 * np.pi**2 * h2  # Clock frequency noise
}

In [ ]:
# Initialize Environment and Spacecraft instances
env = Environment(t_data_array, dates_utc, n_geopot=5)
sat = Spacecraft(t_data_array, rcver_pos, rcver_vel, clock_bias, clock_drift, 
                 mass_func, ignition_func, fthr_dir_func, mass_props)

# Initialize and pre-align GNSS Constellations
gnss_names = ['GPS', 'GLONASS', 'Galileo', 'BDS']
gnss = NavSatSystems(gnss_names)
gnss.load_data(gnss_pos_path, gnss_vis_path)

T_meas = 10  # Measurement interval [s]
t_meas_array = np.arange(t_data_array[0], t_data_array[-1], T_meas)
gnss.prealign_constellations(t_meas_array, eor_time_offset_s)

print("Modules initialized.")

In [ ]:
# Initialize the Filter and the Manager
kf = OrbitalEKF(x0, P0, noise_params)

t_log_array = np.copy(t_data_array)
sim = SimulationManager(kf, env, sat, gnss, t_log_array, t_meas_array)

# Run the simulation
estim_states, P_matrices, mahalanobis_data = sim.run()

In [ ]:
export_path.mkdir(parents=True, exist_ok=True)

# Export States and Covariances (.npy)
np.save(export_path / f"tlog.npy", t_log_array)
np.save(export_path / f"tmeas.npy", t_meas_array)
np.save(export_path / f"states.npy", estim_states)
np.save(export_path / f"covariances.npy", P_matrices)

# Export Mahalanobis Data (.csv)
mah_file = export_path / f'mahalanobis.csv'
with open(mah_file, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['Tiempo[s]', 'DOF', 'Mahalanobis_D2'])
    for row in mahalanobis_data:
        writer.writerow(row)

print(f"Results successfully exported to {export_path}")